In [ ]:
import os

def safe_install(pkg, no_deps=True):
    flag = "--no-deps" if no_deps else ""
    os.system(f"pip install -q {flag} {pkg}")

safe_install("grad-cam", no_deps=False)

import re
import cv2
import copy
import json
import time
import random

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from collections import Counter, defaultdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

import torchvision
from torchvision import models, transforms
import torchvision.transforms.functional as TF

from sklearn.metrics import confusion_matrix, classification_report, f1_score

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if torch.cuda.is_available():
    print(f"GPU          : {torch.cuda.get_device_name(0)}")
    print(f"CUDA Version : {torch.version.cuda}")
    print(f"VRAM         : {torch.cuda.get_device_properties(0).total_memory/1024**3:.2f} GB")
else:
    print("GPU tidak tersedia.")
print(f"\nDevice : {device}")

print("\n===== Library Version =====")
print("NumPy  :", np.__version__)
print("Torch  :", torch.__version__)
print("TVision:", torchvision.__version__)

In [ ]:
CONFIG = {
    'DATA_DIR': '/kaggle/input/datasets/fabiofire/hoya-disease-dataset/data_real_KP',
    'OUTPUT_DIR': '/kaggle/working/outputs_densenet',
    'EXCLUDED_FOLDERS': ['Hoya Compacta (Cadangan)', 'Hoya multiflora (Diganti)', 'Hoya bella (Diganti)'],
    
    'SEED': 42, # Menambahkan SEED untuk menjamin pembagian set data yang konsisten
    
    'NUM_RUNS': 5, 
    'IMG_SIZE': 224, 'BATCH_SIZE': 32, 'NUM_EPOCHS': 50, 'EARLY_STOP_PATIENCE': 15,
    'TRAIN_RATIO': 0.70, 'VAL_RATIO': 0.15, 'TEST_RATIO': 0.15,
    'USE_WEIGHTED_SAMPLER': True, 'USE_RAND_AUGMENT': True,
    'USE_CLAHE': True, 'USE_ADAPTIVE_SHARPEN': True, 'BLUR_THRESHOLD': 100.0, 'SHARPEN_STRENGTH': 1.2,
    
    'DROPOUT': 0.40, 'BACKBONE_LR': 3e-5, 'HEAD_LR': 2e-3, 'WEIGHT_DECAY': 1e-3,
    'BACKBONE': 'densenet121', 
    'USE_MIXUP': True, 'MIXUP_ALPHA': 0.2, 'MIXUP_PROB': 0.5, 'LABEL_SMOOTHING': 0.05,
    'USE_FOCAL_LOSS': True, 'FOCAL_GAMMA': 2.0, 'USE_AMP': True, 'USE_SWA': False,
}

import os
os.makedirs(CONFIG['OUTPUT_DIR'], exist_ok=True)
print("Konfigurasi DenseNet-121 siap.")

In [ ]:
CLASS_ORDER = [
    'Sehat',
    'Bercak Cokelat',
    'Bercak Putih',
    'Daun Layu',
    'Bercak Bintik Hitam',
]

EXCLUDED_DISEASE_CLASSES = ['Bercak Bintik Hitam Putih']

def get_unified_class_name(folder_name):
    lower_name = folder_name.lower()
    if 'sehat' in lower_name:
        return 'Sehat'
    elif ('bintik_hitam' in lower_name or 'bintik-bercak' in lower_name or 'bercak-bintik' in lower_name):
        return 'Bercak Bintik Hitam'
    elif ('bercak_coklat' in lower_name or 'bercak_cokelat' in lower_name):
        return 'Bercak Cokelat'
    elif ('bercak_hitam_putih' in lower_name or 'bercak_putih_hitam' in lower_name or 'bintik_hitam_putih' in lower_name):
        return 'Bercak Bintik Hitam Putih'
    elif 'bercak_putih' in lower_name:
        return 'Bercak Putih'
    elif 'daun_layu' in lower_name:
        return 'Daun Layu'
    return 'Unknown'

def get_base_name(filename):
    name, _ = os.path.splitext(filename)
    name = re.sub(r'_(whitebg|white|putih|bgputih|bg_putih|putihbg|putih_bg)$', '', name, flags=re.IGNORECASE)
    return name.strip()

def is_clean_bg_filename(filename):
    """True = versi background putih (manual), False = versi natural."""
    name, _ = os.path.splitext(filename)
    return bool(re.search(r'_(whitebg|white|putih|bgputih|bg_putih|putihbg|putih_bg)$', name, flags=re.IGNORECASE))

def compute_blur_score(img_path_or_array):
    if isinstance(img_path_or_array, str):
        img = cv2.imread(img_path_or_array)
        if img is None:
            return None
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    else:
        gray = cv2.cvtColor(np.array(img_path_or_array), cv2.COLOR_RGB2GRAY)
    return cv2.Laplacian(gray, cv2.CV_64F).var()

print(f"Kelas dipakai ({len(CLASS_ORDER)}): {CLASS_ORDER}")
print(f"Kelas dihapus: {EXCLUDED_DISEASE_CLASSES}")

In [ ]:
class AdaptiveSharpen:
    def __init__(self, blur_threshold=100.0, sharpen_strength=1.2):
        self.blur_threshold = blur_threshold
        self.sharpen_strength = sharpen_strength

    def __call__(self, img):
        img_np = np.array(img)
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        blur_score = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur_score < self.blur_threshold:
            blurred = cv2.GaussianBlur(img_np, (0, 0), sigmaX=3)
            sharpened = cv2.addWeighted(img_np, 1 + self.sharpen_strength, blurred, -self.sharpen_strength, 0)
            sharpened = np.clip(sharpened, 0, 255).astype(np.uint8)
            return Image.fromarray(sharpened)
        return img


class CLAHETransform:
    def __init__(self, clip_limit=2.0, tile_grid_size=(8, 8)):
        self.clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)

    def __call__(self, img):
        img_np = np.array(img)
        lab = cv2.cvtColor(img_np, cv2.COLOR_RGB2LAB)
        l, a, b = cv2.split(lab)
        l_eq = self.clahe.apply(l)
        lab_eq = cv2.merge((l_eq, a, b))
        img_eq = cv2.cvtColor(lab_eq, cv2.COLOR_LAB2RGB)
        return Image.fromarray(img_eq)


class DiscreteRotation:
    def __init__(self, angles):
        self.angles = angles

    def __call__(self, img):
        angle = random.choice(self.angles)
        return TF.rotate(img, angle)

print("AdaptiveSharpen, CLAHE & DiscreteRotation siap.")

In [ ]:
class HoyaLeafDataset(Dataset):
    def __init__(self, root_dir, class_order, excluded_disease_classes=None, excluded_folders=None):
        self.root_dir = root_dir
        self.classes = class_order
        self.excluded_disease_classes = excluded_disease_classes or []
        self.excluded_folders = excluded_folders or []

        self.image_paths = []
        self.disease_labels = []
        self.species_labels = []
        self.species_list = []
        self.leaf_ids = []
        self.is_clean_version = []

        species_found = set()
        for species in sorted(os.listdir(root_dir)):
            if species in self.excluded_folders:
                continue
            if os.path.isdir(os.path.join(root_dir, species)):
                species_found.add(species)
        self.species_names = sorted(species_found)
        species_to_idx = {s: i for i, s in enumerate(self.species_names)}

        skipped_unknown = []
        skipped_excluded_class = defaultdict(int)

        for species in self.species_names:
            species_path = os.path.join(root_dir, species)
            for disease_raw in os.listdir(species_path):
                disease_path = os.path.join(species_path, disease_raw)
                if not os.path.isdir(disease_path):
                    continue
                disease_unified = get_unified_class_name(disease_raw)
                if disease_unified in self.excluded_disease_classes:
                    n_imgs = sum(1 for f in os.listdir(disease_path) if f.lower().endswith(('.jpg', '.jpeg', '.png')))
                    skipped_excluded_class[disease_unified] += n_imgs
                    continue
                if disease_unified == 'Unknown':
                    skipped_unknown.append(os.path.join(species, disease_raw))
                    continue
                disease_idx = self.classes.index(disease_unified)
                species_idx = species_to_idx[species]

                for img_name in os.listdir(disease_path):
                    if img_name.lower().endswith(('.jpg', '.jpeg', '.png')):
                        img_path = os.path.join(disease_path, img_name)
                        self.image_paths.append(img_path)
                        self.disease_labels.append(disease_idx)
                        self.species_labels.append(species_idx)
                        self.species_list.append(species)
                        self.leaf_ids.append(f"{species}_{get_base_name(img_name)}")
                        self.is_clean_version.append(is_clean_bg_filename(img_name))

        if skipped_unknown:
            print(f"[PERINGATAN] {len(skipped_unknown)} folder dilewati")
        if skipped_excluded_class:
            print("[INFO] Kelas dihapus dari dataset:")
            for cn, n in skipped_excluded_class.items():
                print(f"   - {cn}: {n} gambar dilewati")

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        return image, self.disease_labels[idx], self.species_labels[idx], self.image_paths[idx], self.leaf_ids[idx]


full_dataset = HoyaLeafDataset(
    CONFIG['DATA_DIR'], class_order=CLASS_ORDER,
    excluded_disease_classes=EXCLUDED_DISEASE_CLASSES,
    excluded_folders=CONFIG['EXCLUDED_FOLDERS'],
)
class_names = full_dataset.classes
species_names = full_dataset.species_names

n_clean = sum(full_dataset.is_clean_version)
n_natural = len(full_dataset) - n_clean
print(f"Total gambar: {len(full_dataset)} | Kelas: {len(class_names)} | Spesies: {len(species_names)}")
print(f"Versi background putih (manual): {n_clean} | Versi natural: {n_natural}")

In [ ]:
def print_and_plot_distribution(labels, names, title, save_name=None):
    counts = np.bincount(labels, minlength=len(names))
    print(f"--- {title} ---")
    for cn, c in zip(names, counts):
        print(f"  {cn:22s}: {c}")
    plt.figure(figsize=(8, 4))
    bars = plt.barh(names, counts, color='steelblue')
    plt.title(title)
    plt.xlabel('Jumlah Gambar')
    for bar, c in zip(bars, counts):
        plt.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, str(c), va='center')
    plt.tight_layout()
    if save_name:
        plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], save_name), dpi=150, bbox_inches='tight')
    plt.show()
    return counts

_ = print_and_plot_distribution(full_dataset.disease_labels, class_names, 'Distribusi Penyakit - Seluruh Data', 'distribusi_penyakit.png')
_ = print_and_plot_distribution(full_dataset.species_labels, species_names, 'Distribusi Spesies - Seluruh Data', 'distribusi_spesies.png')

In [ ]:
leaf_to_disease = {}
leaf_to_species = {}
leaf_to_indices = defaultdict(list)
for i, leaf_id in enumerate(full_dataset.leaf_ids):
    leaf_to_indices[leaf_id].append(i)
    leaf_to_disease[leaf_id] = full_dataset.disease_labels[i]
    leaf_to_species[leaf_id] = full_dataset.species_labels[i]

leaves_per_group = defaultdict(list)
for leaf_id in leaf_to_disease:
    key = (leaf_to_disease[leaf_id], leaf_to_species[leaf_id])
    leaves_per_group[key].append(leaf_id)

random.seed(CONFIG['SEED'])
train_leaves, val_leaves, test_leaves = set(), set(), set()

for key, leaves in leaves_per_group.items():
    leaves = leaves.copy()
    random.shuffle(leaves)
    n = len(leaves)
    n_train = max(1, int(round(CONFIG['TRAIN_RATIO'] * n)))
    n_val = max(1, int(round(CONFIG['VAL_RATIO'] * n))) if n - n_train > 1 else 0
    n_train = min(n_train, n)
    n_val = min(n_val, max(0, n - n_train))
    train_leaves.update(leaves[:n_train])
    val_leaves.update(leaves[n_train:n_train + n_val])
    test_leaves.update(leaves[n_train + n_val:])

train_indices, val_indices, test_indices = [], [], []
for leaf_id, idxs in leaf_to_indices.items():
    if leaf_id in train_leaves:
        train_indices.extend(idxs)
    elif leaf_id in val_leaves:
        val_indices.extend(idxs)
    else:
        test_indices.extend(idxs)

print(f"Sebelum filter -> Train: {len(train_indices)} | Val: {len(val_indices)} | Test: {len(test_indices)}")

In [ ]:
# Val & Test HARUS pakai versi natural saja -> evaluasi mencerminkan kondisi nyata (webcam/upload)
val_indices = [i for i in val_indices if not full_dataset.is_clean_version[i]]
test_indices = [i for i in test_indices if not full_dataset.is_clean_version[i]]

print(f"Setelah filter  -> Train: {len(train_indices)} (natural+putih) | Val: {len(val_indices)} (natural) | Test: {len(test_indices)} (natural)")

train_disease = [full_dataset.disease_labels[i] for i in train_indices]
train_species = [full_dataset.species_labels[i] for i in train_indices]
val_disease = [full_dataset.disease_labels[i] for i in val_indices]
val_species = [full_dataset.species_labels[i] for i in val_indices]
test_disease = [full_dataset.disease_labels[i] for i in test_indices]
test_species = [full_dataset.species_labels[i] for i in test_indices]

disease_counts_train = np.bincount(train_disease, minlength=len(class_names))
species_counts_train = np.bincount(train_species, minlength=len(species_names))

for name, labels in [('TRAIN Penyakit', train_disease), ('VAL Penyakit', val_disease), ('TEST Penyakit', test_disease)]:
    _ = print_and_plot_distribution(labels, class_names, f'Distribusi - {name}')
for name, labels in [('TRAIN Spesies', train_species), ('VAL Spesies', val_species), ('TEST Spesies', test_species)]:
    _ = print_and_plot_distribution(labels, species_names, f'Distribusi - {name}')

In [ ]:
preprocess_steps = []
if CONFIG['USE_ADAPTIVE_SHARPEN']:
    preprocess_steps.append(AdaptiveSharpen(blur_threshold=CONFIG['BLUR_THRESHOLD'], sharpen_strength=CONFIG['SHARPEN_STRENGTH']))
if CONFIG['USE_CLAHE']:
    preprocess_steps.append(CLAHETransform())

train_transforms_list = [
    transforms.Resize((int(CONFIG['IMG_SIZE'] * 1.15), int(CONFIG['IMG_SIZE'] * 1.15))),
    *preprocess_steps,
    DiscreteRotation([0, 90, 180, 270]),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3)
]

if CONFIG.get('USE_RAND_AUGMENT', False):
    train_transforms_list.append(transforms.RandAugment(num_ops=2, magnitude=9))

train_transforms_list.extend([
    transforms.RandomResizedCrop(CONFIG['IMG_SIZE'], scale=(0.75, 1.0)),
    transforms.ColorJitter(brightness=0.15, contrast=0.10, saturation=0.08),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.08), ratio=(0.3, 3.3)),
])

train_transform = transforms.Compose(train_transforms_list)

eval_transform = transforms.Compose([
    transforms.Resize((CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'])),
    *preprocess_steps,
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])

print("Pipeline transform siap dengan RandAugment.")

In [ ]:
class CustomSubset(Dataset):
    def __init__(self, dataset, indices, transform=None):
        self.dataset = dataset
        self.indices = indices
        self.transform = transform

    def __getitem__(self, idx):
        raw_img, disease_label, species_label, path, leaf_id = self.dataset[self.indices[idx]]
        img = self.transform(raw_img) if self.transform else raw_img
        display_img = np.array(raw_img.resize((256, 256)))
        return img, disease_label, species_label, path, display_img

    def __len__(self):
        return len(self.indices)


image_datasets = {
    'train': CustomSubset(full_dataset, train_indices, transform=train_transform),
    'val': CustomSubset(full_dataset, val_indices, transform=eval_transform),
    'test': CustomSubset(full_dataset, test_indices, transform=eval_transform),
}

if CONFIG['USE_WEIGHTED_SAMPLER']:
    disease_w = 1.0 / np.maximum(disease_counts_train, 1)
    species_w = 1.0 / np.maximum(species_counts_train, 1)
    disease_w = disease_w / disease_w.sum() * len(class_names)
    species_w = species_w / species_w.sum() * len(species_names)
    combined_weights = [disease_w[d] * species_w[s] for d, s in zip(train_disease, train_species)]

    max_class_count = int(max(disease_counts_train))
    total_balanced_samples = int(max_class_count * len(class_names))
    sampler = WeightedRandomSampler(weights=combined_weights, num_samples=total_balanced_samples, replacement=True)
    train_loader = DataLoader(image_datasets['train'], batch_size=CONFIG['BATCH_SIZE'], sampler=sampler,
                               num_workers=2, pin_memory=True)
    print("WeightedRandomSampler GABUNGAN (disease x species) aktif.")
else:
    total_balanced_samples = len(image_datasets['train'])
    train_loader = DataLoader(image_datasets['train'], batch_size=CONFIG['BATCH_SIZE'], shuffle=True,
                               num_workers=2, pin_memory=True)

dataloaders = {
    'train': train_loader,
    'val': DataLoader(image_datasets['val'], batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=2, pin_memory=True),
    'test': DataLoader(image_datasets['test'], batch_size=CONFIG['BATCH_SIZE'], shuffle=False, num_workers=2, pin_memory=True),
}
dataset_sizes = {'train': total_balanced_samples, 'val': len(image_datasets['val']), 'test': len(image_datasets['test'])}
print(dataset_sizes)

In [ ]:
samples_per_class = {}
random.seed(123)
indices_shuffled = list(range(len(image_datasets['train'])))
random.shuffle(indices_shuffled)
for idx in indices_shuffled:
    _, disease_label, _, _, _ = full_dataset[image_datasets['train'].indices[idx]]
    if disease_label not in samples_per_class:
        samples_per_class[disease_label] = idx
    if len(samples_per_class) == len(class_names):
        break

sorted_labels = sorted(samples_per_class.keys())
fig, axes = plt.subplots(len(sorted_labels), 2, figsize=(8, 4 * len(sorted_labels)))
fig.suptitle('Visualisasi: Original vs Hasil Augmentasi', fontsize=15, fontweight='bold')
for row, label in enumerate(sorted_labels):
    idx = samples_per_class[label]
    img_tensor, d_lbl, s_lbl, path, display_img = image_datasets['train'][idx]
    axes[row, 0].imshow(display_img)
    axes[row, 0].set_title(f'Asli\n{class_names[d_lbl]}')
    axes[row, 0].axis('off')
    img_disp = img_tensor.numpy().transpose(1, 2, 0)
    img_disp = IMAGENET_STD * img_disp + IMAGENET_MEAN
    img_disp = np.clip(img_disp, 0, 1)
    axes[row, 1].imshow(img_disp)
    axes[row, 1].set_title('Telah Di-Augmentasi')
    axes[row, 1].axis('off')
plt.tight_layout(); plt.subplots_adjust(top=0.95)
plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], 'augmentasi_before_after.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        self.fc1 = nn.Conv2d(channels, channels // reduction, 1, bias=False)
        self.relu = nn.ReLU(inplace=True)
        self.fc2 = nn.Conv2d(channels // reduction, channels, 1, bias=False)
        self.sigmoid_channel = nn.Sigmoid()
        
        self.conv_spatial = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid_spatial = nn.Sigmoid()

    def forward(self, x):
        avg_out = self.fc2(self.relu(self.fc1(F.adaptive_avg_pool2d(x, 1))))
        max_out = self.fc2(self.relu(self.fc1(F.adaptive_max_pool2d(x, 1))))
        out = avg_out + max_out
        x = x * self.sigmoid_channel(out)
        
        avg_out = torch.mean(x, dim=1, keepdim=True)
        max_out, _ = torch.max(x, dim=1, keepdim=True)
        spatial_out = self.sigmoid_spatial(self.conv_spatial(torch.cat([avg_out, max_out], dim=1)))
        return x * spatial_out

class MultiTaskHoyaModel(nn.Module):
    def __init__(self, feature_extractor, in_features, num_disease, num_species, dropout):
        super().__init__()
        self.feature_extractor = feature_extractor
        self.attention = CBAM(in_features)
        self.disease_head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(in_features, 512), nn.GELU(),
            nn.BatchNorm1d(512), nn.Dropout(dropout), nn.Linear(512, num_disease),
        )
        self.species_head = nn.Sequential(
            nn.Dropout(dropout), nn.Linear(in_features, 256), nn.GELU(),
            nn.BatchNorm1d(256), nn.Dropout(dropout), nn.Linear(256, num_species),
        )

    def forward(self, x):
        feat_map = self.feature_extractor(x)
        feat_map = self.attention(feat_map) 
        feat = F.relu(feat_map, inplace=True)
        feat = F.adaptive_avg_pool2d(feat, (1, 1)).view(feat.size(0), -1)
        return self.disease_head(feat), self.species_head(feat)

def build_model(num_disease, num_species, dropout=0.40, unfreeze='block4', backbone='densenet121'):
    base = models.densenet121(weights=models.DenseNet121_Weights.IMAGENET1K_V1)
    in_features = base.classifier.in_features
    feature_extractor = base.features

    for name, param in feature_extractor.named_parameters():
        param.requires_grad = False
        
    for name, param in feature_extractor.named_parameters():
        if name.startswith('denseblock4') or name.startswith('norm5'):
            param.requires_grad = True

    model = MultiTaskHoyaModel(feature_extractor, in_features, num_disease, num_species, dropout)
    return model.to(device)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, weight=None, gamma=2.0, label_smoothing=0.0):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, weight=self.weight, label_smoothing=self.label_smoothing, reduction='none')
        pt = torch.exp(-ce_loss)
        return (((1 - pt) ** self.gamma) * ce_loss).mean()

disease_weights = torch.tensor(1.0 / np.maximum(disease_counts_train, 1), dtype=torch.float32)
disease_weights = (disease_weights / disease_weights.sum() * len(class_names)).to(device)
species_weights = torch.tensor(1.0 / np.maximum(species_counts_train, 1), dtype=torch.float32)
species_weights = (species_weights / species_weights.sum() * len(species_names)).to(device)

if CONFIG.get('USE_FOCAL_LOSS', True):
    criterion_disease = FocalLoss(weight=disease_weights, gamma=CONFIG['FOCAL_GAMMA'], label_smoothing=CONFIG['LABEL_SMOOTHING'])
    criterion_species = FocalLoss(weight=species_weights, gamma=CONFIG['FOCAL_GAMMA'], label_smoothing=CONFIG['LABEL_SMOOTHING'])
    print("Menggunakan Focal Loss.")
else:
    criterion_disease = nn.CrossEntropyLoss(weight=disease_weights, label_smoothing=CONFIG['LABEL_SMOOTHING'])
    criterion_species = nn.CrossEntropyLoss(weight=species_weights, label_smoothing=CONFIG['LABEL_SMOOTHING'])

DISEASE_LOSS_WEIGHT = 1.0
SPECIES_LOSS_WEIGHT = 0.5

In [ ]:
def mixup_data(x, y_disease, y_species, alpha=0.2):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1
    index = torch.randperm(x.size(0), device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y_disease, y_disease[index], y_species, y_species[index], lam


def cutmix_data(x, y_disease, y_species, alpha=1.0):
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    H, W = x.size(2), x.size(3)
    cut_ratio = np.sqrt(1.0 - lam)
    cut_h, cut_w = int(H * cut_ratio), int(W * cut_ratio)
    cy, cx = np.random.randint(H), np.random.randint(W)
    y1, y2 = np.clip(cy - cut_h // 2, 0, H), np.clip(cy + cut_h // 2, 0, H)
    x1, x2 = np.clip(cx - cut_w // 2, 0, W), np.clip(cx + cut_w // 2, 0, W)
    x_cut = x.clone()
    x_cut[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam_adjusted = 1 - ((x2 - x1) * (y2 - y1) / (H * W))
    return x_cut, y_disease, y_disease[index], y_species, y_species[index], lam_adjusted


CHECKPOINT_PATH = os.path.join(CONFIG['OUTPUT_DIR'], f'checkpoint_{CONFIG["BACKBONE"]}.pth')
SAVE_EVERY_N_EPOCH = 3


def train_model(model, dataloaders, dataset_sizes, criterion_d, criterion_s, dw, sw,
                 optimizer, scheduler, num_epochs, patience, unfreeze_schedule=None,
                 use_mixup=False, mixup_alpha=0.2, mixup_prob=0.5, use_cutmix=True, use_amp=True,
                 selection_weight_disease=0.7, selection_weight_species=0.3,
                 checkpoint_path=None, save_every=3):
    since = time.time()
    unfreeze_schedule = unfreeze_schedule or {0: 'head_only'}
    scaler = torch.amp.GradScaler('cuda', enabled=(use_amp and torch.cuda.is_available()))

    start_epoch = 0
    best_model_wts = copy.deepcopy(model.state_dict())
    best_combined_score = 0.0
    best_f1_disease_at_best = 0.0
    epochs_no_improve = 0
    history = {'train_loss': [], 'val_loss': [], 'val_f1_disease': [], 'val_f1_species': [], 'val_combined_score': []}

    if checkpoint_path and os.path.exists(checkpoint_path):
        print(f"Checkpoint ditemukan di {checkpoint_path}, melanjutkan training...")
        ckpt = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(ckpt['model_state'])
        optimizer.load_state_dict(ckpt['optimizer_state'])
        scheduler.load_state_dict(ckpt['scheduler_state'])
        start_epoch = ckpt['epoch'] + 1
        history = ckpt['history']
        best_model_wts = ckpt['best_model_wts']
        best_combined_score = ckpt['best_combined_score']
        best_f1_disease_at_best = ckpt.get('best_f1_disease_at_best', 0.0)
        epochs_no_improve = ckpt['epochs_no_improve']
        print(f"Melanjutkan dari epoch {start_epoch+1}, best combined score: {best_combined_score:.4f}")
        for ep, stage in sorted(unfreeze_schedule.items()):
            if ep <= start_epoch - 1:
                set_unfreeze_stage(model, stage)

    for epoch in range(start_epoch, num_epochs):
        if epoch in unfreeze_schedule:
            stage = unfreeze_schedule[epoch]
            set_unfreeze_stage(model, stage)
            n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"[Unfreeze] Epoch {epoch+1}: stage='{stage}' -> {n_train:,} param trainable")

        print(f'Epoch {epoch + 1}/{num_epochs}'); print('-' * 30)

        for phase in ['train', 'val']:
            model.train() if phase == 'train' else model.eval()
            running_loss = 0.0
            preds_d, labels_d, preds_s, labels_s = [], [], [], []

            for inputs, disease_lbl, species_lbl, _, _ in dataloaders[phase]:
                inputs = inputs.to(device, non_blocking=True)
                disease_lbl = disease_lbl.to(device, non_blocking=True)
                species_lbl = species_lbl.to(device, non_blocking=True)
                optimizer.zero_grad()

                apply_mixup = (phase == 'train' and use_mixup and random.random() < mixup_prob)
                use_cutmix_this_batch = apply_mixup and use_cutmix and random.random() < 0.5

                with torch.set_grad_enabled(phase == 'train'):
                    with torch.amp.autocast('cuda', enabled=(use_amp and torch.cuda.is_available())):
                        if apply_mixup:
                            if use_cutmix_this_batch:
                                inputs_mix, da, db, sa, sb, lam = cutmix_data(inputs, disease_lbl, species_lbl, alpha=1.0)
                            else:
                                inputs_mix, da, db, sa, sb, lam = mixup_data(inputs, disease_lbl, species_lbl, mixup_alpha)
                            out_d, out_s = model(inputs_mix)
                            loss_d = lam * criterion_d(out_d, da) + (1 - lam) * criterion_d(out_d, db)
                            loss_s = lam * criterion_s(out_s, sa) + (1 - lam) * criterion_s(out_s, sb)
                        else:
                            out_d, out_s = model(inputs)
                            loss_d = criterion_d(out_d, disease_lbl)
                            loss_s = criterion_s(out_s, species_lbl)
                        loss = dw * loss_d + sw * loss_s
                        pred_d = torch.argmax(out_d, 1)
                        pred_s = torch.argmax(out_s, 1)

                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                running_loss += loss.item() * inputs.size(0)
                if not apply_mixup:
                    preds_d.extend(pred_d.cpu().numpy()); labels_d.extend(disease_lbl.cpu().numpy())
                    preds_s.extend(pred_s.cpu().numpy()); labels_s.extend(species_lbl.cpu().numpy())

            epoch_loss = running_loss / dataset_sizes[phase]
            f1_d = f1_score(labels_d, preds_d, average='macro', zero_division=0) if labels_d else float('nan')
            f1_s = f1_score(labels_s, preds_s, average='macro', zero_division=0) if labels_s else float('nan')
            print(f'{phase.capitalize():5s} Loss: {epoch_loss:.4f}  F1-Disease: {f1_d:.4f}  F1-Species: {f1_s:.4f}')

            if phase == 'train':
                history['train_loss'].append(epoch_loss)
            else:
                history['val_loss'].append(epoch_loss)
                history['val_f1_disease'].append(f1_d)
                history['val_f1_species'].append(f1_s)
                combined_score = selection_weight_disease * f1_d + selection_weight_species * f1_s
                history['val_combined_score'].append(combined_score)
                scheduler.step(epoch_loss)

                if combined_score > best_combined_score:
                    best_combined_score = combined_score
                    best_f1_disease_at_best = f1_d
                    best_model_wts = copy.deepcopy(model.state_dict())
                    epochs_no_improve = 0
                    print(f'  -> Model terbaik baru (Skor gabungan: {best_combined_score:.4f})')
                else:
                    epochs_no_improve += 1
        print()

        if checkpoint_path and ((epoch + 1) % save_every == 0 or epoch == num_epochs - 1):
            torch.save({
                'epoch': epoch, 'model_state': model.state_dict(),
                'optimizer_state': optimizer.state_dict(), 'scheduler_state': scheduler.state_dict(),
                'history': history, 'best_model_wts': best_model_wts,
                'best_combined_score': best_combined_score,
                'best_f1_disease_at_best': best_f1_disease_at_best,
                'epochs_no_improve': epochs_no_improve,
            }, checkpoint_path)
            print(f"  [Checkpoint disimpan di epoch {epoch+1}]")

        if epochs_no_improve >= patience:
            print(f'Early stopping di epoch {epoch + 1}')
            break

    print(f'\nTraining selesai dalam {(time.time()-since)//60:.0f}m {(time.time()-since)%60:.0f}s')
    print(f'Best Combined Score: {best_combined_score:.4f} (F1-Disease saat itu: {best_f1_disease_at_best:.4f})')
    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
import math
import copy

all_runs_metrics = []

for run_id in range(1, CONFIG['NUM_RUNS'] + 1):
    print(f"\n{'='*50}\nMEMULAI RUN #{run_id} ({CONFIG['BACKBONE']})\n{'='*50}")
    
    set_seed(42 + run_id)
    
    model_ft = build_model(len(class_names), len(species_names), dropout=CONFIG['DROPOUT'], unfreeze='block4', backbone=CONFIG['BACKBONE'])
    
    params_backbone = [p for n, p in model_ft.named_parameters() if p.requires_grad and n.startswith('feature_extractor')]
    params_head = [p for n, p in model_ft.named_parameters() if p.requires_grad and not n.startswith('feature_extractor')]
    
    optimizer_ft = optim.AdamW([
        {'params': params_backbone, 'lr': CONFIG['BACKBONE_LR'], 'weight_decay': CONFIG['WEIGHT_DECAY']},
        {'params': params_head, 'lr': CONFIG['HEAD_LR'], 'weight_decay': CONFIG['WEIGHT_DECAY']},
    ])
    
    steps_per_epoch = len(dataloaders['train'])
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer_ft, 
        max_lr=[CONFIG['BACKBONE_LR'] * 5, CONFIG['HEAD_LR'] * 2], 
        epochs=CONFIG['NUM_EPOCHS'], 
        steps_per_epoch=steps_per_epoch,
        pct_start=0.2, div_factor=10, final_div_factor=1e3
    )

    best_combined_score = 0.0
    best_f1_d = 0.0
    best_model_wts = copy.deepcopy(model_ft.state_dict())
    
    for epoch in range(CONFIG['NUM_EPOCHS']):
        print(f"Run {run_id} | Epoch {epoch+1}/{CONFIG['NUM_EPOCHS']}")
        
        for phase in ['train', 'val']:
            model_ft.train() if phase == 'train' else model_ft.eval()
            running_loss = 0.0
            preds_d, labels_d, preds_s, labels_s = [], [], [], []
            
            for inputs, disease_lbl, species_lbl, _, _ in dataloaders[phase]:
                inputs = inputs.to(device)
                disease_lbl = disease_lbl.to(device)
                species_lbl = species_lbl.to(device)
                optimizer_ft.zero_grad()
                
                apply_mixup = (phase == 'train' and CONFIG['USE_MIXUP'] and random.random() < CONFIG['MIXUP_PROB'])
                
                with torch.set_grad_enabled(phase == 'train'):
                    # PENGGUNAAN AUTOCAST YANG BENAR (WARNING HILANG)
                    with torch.amp.autocast('cuda', enabled=CONFIG['USE_AMP']):
                        if apply_mixup:
                            inputs_mix, da, db, sa, sb, lam = mixup_data(inputs, disease_lbl, species_lbl, CONFIG['MIXUP_ALPHA'])
                            out_d, out_s = model_ft(inputs_mix)
                            loss_d = lam * criterion_disease(out_d, da) + (1 - lam) * criterion_disease(out_d, db)
                            loss_s = lam * criterion_species(out_s, sa) + (1 - lam) * criterion_species(out_s, sb)
                        else:
                            out_d, out_s = model_ft(inputs)
                            loss_d = criterion_disease(out_d, disease_lbl)
                            loss_s = criterion_species(out_s, species_lbl)
                        
                        loss = DISEASE_LOSS_WEIGHT * loss_d + SPECIES_LOSS_WEIGHT * loss_s
                        pred_d = torch.argmax(out_d, 1)
                        pred_s = torch.argmax(out_s, 1)
                        
                    if phase == 'train':
                        loss.backward()
                        optimizer_ft.step()
                        scheduler.step()

                running_loss += loss.item() * inputs.size(0)
                if not apply_mixup:
                    preds_d.extend(pred_d.cpu().numpy()); labels_d.extend(disease_lbl.cpu().numpy())
                    preds_s.extend(pred_s.cpu().numpy()); labels_s.extend(species_lbl.cpu().numpy())

            epoch_loss = running_loss / dataset_sizes[phase]
            f1_d = f1_score(labels_d, preds_d, average='macro', zero_division=0) if labels_d else 0.0
            f1_s = f1_score(labels_s, preds_s, average='macro', zero_division=0) if labels_s else 0.0
            
            if phase == 'val':
                combined_score = 0.7 * f1_d + 0.3 * f1_s
                if combined_score > best_combined_score:
                    best_combined_score = combined_score
                    best_f1_d = f1_d
                    best_model_wts = copy.deepcopy(model_ft.state_dict())
                    print(f" -> Best Score: {best_combined_score:.4f} (F1-D: {f1_d:.4f})")

    model_ft.load_state_dict(best_model_wts)
    model_ft.eval()
    t_preds_d, t_labels_d = [], []
    with torch.no_grad():
        for inputs, disease_lbl, _, _, _ in dataloaders['test']:
            out_d, _ = model_ft(inputs.to(device))
            t_preds_d.extend(torch.argmax(out_d, 1).cpu().numpy())
            t_labels_d.extend(disease_lbl.numpy())
            
    test_acc = sum(np.array(t_preds_d) == np.array(t_labels_d)) / len(t_labels_d)
    test_f1 = f1_score(t_labels_d, t_preds_d, average='macro')
    
    print(f"\n--- HASIL RUN #{run_id} ---")
    print(f"Test Accuracy (Disease): {test_acc:.4f} | Test F1-Macro: {test_f1:.4f}")
    
    all_runs_metrics.append({'Run': run_id, 'Acc': test_acc, 'F1': test_f1, 'model_wts': best_model_wts})
    
accs = [m['Acc'] for m in all_runs_metrics]
f1s = [m['F1'] for m in all_runs_metrics]
print(f"\n{'='*50}")
print(f"KESIMPULAN 5 RUNS ({CONFIG['BACKBONE']})")
print(f"Rata-rata Akurasi Penyakit: {np.mean(accs):.4f} ± {np.std(accs):.4f}")
print(f"Rata-rata F1-Macro Penyakit: {np.mean(f1s):.4f} ± {np.std(f1s):.4f}")

best_run = max(all_runs_metrics, key=lambda x: x['F1'])
print(f"\nMenggunakan bobot dari Run #{best_run['Run']} untuk Evaluasi dan Grad-CAM selanjutnya.")
model_ft.load_state_dict(best_run['model_wts'])

In [ ]:
from torch.optim.swa_utils import AveragedModel, update_bn

if CONFIG['USE_SWA']:
    print(f"\n=== Fine-tuning SWA ({CONFIG['SWA_EPOCHS']} epoch tambahan) ===")
    swa_model = AveragedModel(model_ft)
    swa_optimizer = optim.AdamW(model_ft.parameters(), lr=CONFIG['SWA_LR'], weight_decay=1e-4)

    model_ft.train()
    for epoch in range(CONFIG['SWA_EPOCHS']):
        for inputs, labels_disease, labels_species, _, _ in dataloaders['train']:
            inputs = inputs.to(device)
            labels_disease = labels_disease.to(device)
            labels_species = labels_species.to(device)
            swa_optimizer.zero_grad()
            outputs_disease, outputs_species = model_ft(inputs)
            loss_disease = criterion_disease(outputs_disease, labels_disease)
            loss_species = criterion_species(outputs_species, labels_species)
            loss = DISEASE_LOSS_WEIGHT * loss_disease + SPECIES_LOSS_WEIGHT * loss_species
            loss.backward()
            swa_optimizer.step()
        swa_model.update_parameters(model_ft)
        print(f"  SWA epoch {epoch+1}/{CONFIG['SWA_EPOCHS']} selesai")

    update_bn(dataloaders['train'], swa_model, device=device)

    def quick_eval_combined(m, loader, w_disease=0.7, w_species=0.3):
        m.eval()
        preds_d, labels_d, preds_s, labels_s = [], [], [], []
        with torch.no_grad():
            for inputs, labels_disease, labels_species, _, _ in loader:
                inputs = inputs.to(device)
                outputs_disease, outputs_species = m(inputs)
                preds_d.extend(torch.argmax(outputs_disease, dim=1).cpu().numpy())
                labels_d.extend(labels_disease.numpy())
                preds_s.extend(torch.argmax(outputs_species, dim=1).cpu().numpy())
                labels_s.extend(labels_species.numpy())
        f1_d = f1_score(labels_d, preds_d, average='macro', zero_division=0)
        f1_s = f1_score(labels_s, preds_s, average='macro', zero_division=0)
        return w_disease * f1_d + w_species * f1_s, f1_d, f1_s

    score_normal, f1d_normal, f1s_normal = quick_eval_combined(model_ft, dataloaders['val'])
    score_swa, f1d_swa, f1s_swa = quick_eval_combined(swa_model, dataloaders['val'])
    print(f"\nModel biasa -> Skor: {score_normal:.4f} (D:{f1d_normal:.4f}, S:{f1s_normal:.4f})")
    print(f"SWA model   -> Skor: {score_swa:.4f} (D:{f1d_swa:.4f}, S:{f1s_swa:.4f})")

    if score_swa > score_normal:
        print("-> SWA lebih baik, dipakai sebagai model final.")
        model_ft = swa_model.module
    else:
        print("-> Model biasa masih lebih baik, SWA TIDAK dipakai.")
else:
    print("USE_SWA=False, cell SWA dilewati.")

In [ ]:
def evaluate_with_tta(model, dataloader, device):
    model.eval()
    preds_d, labels_d, preds_s, labels_s = [], [], [], []
    with torch.no_grad():
        for inputs, disease_lbl, species_lbl, _, _ in dataloader:
            inputs = inputs.to(device)
            out_d1, out_s1 = model(inputs)
            out_d2, out_s2 = model(torch.flip(inputs, dims=[3]))
            out_d3, out_s3 = model(torch.flip(inputs, dims=[2]))
            out_d4, out_s4 = model(TF.rotate(inputs, 90))
            out_d5, out_s5 = model(TF.rotate(inputs, 270))
            probs_d = sum(F.softmax(o, 1) for o in [out_d1, out_d2, out_d3, out_d4, out_d5]) / 5
            probs_s = sum(F.softmax(o, 1) for o in [out_s1, out_s2, out_s3, out_s4, out_s5]) / 5
            preds_d.extend(torch.argmax(probs_d, 1).cpu().numpy()); labels_d.extend(disease_lbl.numpy())
            preds_s.extend(torch.argmax(probs_s, 1).cpu().numpy()); labels_s.extend(species_lbl.numpy())
    return labels_d, preds_d, labels_s, preds_s


labels_d, preds_d, labels_s, preds_s = evaluate_with_tta(model_ft, dataloaders['test'], device)

report_dict_disease = classification_report(labels_d, preds_d, target_names=class_names, output_dict=True, zero_division=0)
report_dict_species = classification_report(labels_s, preds_s, target_names=species_names, output_dict=True, zero_division=0)

print("=== CLASSIFICATION REPORT — PENYAKIT ===")
print(classification_report(labels_d, preds_d, target_names=class_names, zero_division=0))
print("\n=== CLASSIFICATION REPORT — SPESIES ===")
print(classification_report(labels_s, preds_s, target_names=species_names, zero_division=0))

cm_d = confusion_matrix(labels_d, preds_d)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_d, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
plt.title(f'Confusion Matrix Penyakit — {CONFIG["BACKBONE"].upper()}')
plt.tight_layout(); plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], f'cm_disease_{CONFIG["BACKBONE"]}.png'), dpi=150); plt.show()

cm_s = confusion_matrix(labels_s, preds_s)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_s, annot=True, fmt='d', cmap='Greens', xticklabels=species_names, yticklabels=species_names)
plt.title(f'Confusion Matrix Spesies — {CONFIG["BACKBONE"].upper()}')
plt.tight_layout(); plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], f'cm_species_{CONFIG["BACKBONE"]}.png'), dpi=150); plt.show()

In [ ]:
def plot_metrics_per_class(report_dict, names, title_suffix):
    precisions = [report_dict[c]['precision'] * 100 for c in names]
    recalls = [report_dict[c]['recall'] * 100 for c in names]
    f1s = [report_dict[c]['f1-score'] * 100 for c in names]
    fig, axes = plt.subplots(1, 3, figsize=(18, 0.55 * len(names) + 2))
    for ax, (title, values, color) in zip(axes, [('Precision', precisions, 'steelblue'), ('Recall', recalls, 'darkorange'), ('F1-Score', f1s, 'seagreen')]):
        bars = ax.barh(names, values, color=color)
        ax.set_title(title); ax.set_xlim(0, 100)
        for bar, v in zip(bars, values):
            ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2, f'{v:.1f}', va='center')
    plt.suptitle(f'Metrik Per Kelas — {title_suffix}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], f'metrics_{title_suffix.lower().replace(" ", "_")}.png'), dpi=150, bbox_inches='tight')
    plt.show()

plot_metrics_per_class(report_dict_disease, class_names, f'Penyakit - {CONFIG["BACKBONE"].upper()}')
plot_metrics_per_class(report_dict_species, species_names, f'Spesies - {CONFIG["BACKBONE"].upper()}')

In [ ]:
class DiseaseOnlyWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def forward(self, x):
        disease_out, _ = self.model(x)
        return disease_out

model_ft.eval()
cam_model = DiseaseOnlyWrapper(model_ft)

# KUNCI PERUBAHAN: Arahkan ke modul Attention (berlaku untuk ResNet maupun DenseNet)
target_layers = [model_ft.attention] 
cam = GradCAM(model=cam_model, target_layers=target_layers)

inputs, disease_lbl, species_lbl, paths, display_imgs = next(iter(dataloaders['test']))
inputs_gpu = inputs.to(device)
n_show = min(6, len(inputs))
fig, axes = plt.subplots(n_show, 3, figsize=(13, 4 * n_show))
fig.suptitle(f'Grad-CAM ({CONFIG["BACKBONE"]}) — Fokus HANYA pada Area Gejala Penyakit', fontsize=15, fontweight='bold')

with torch.no_grad():
    out_d, out_s = model_ft(inputs_gpu)
    conf_d, pred_d = torch.max(F.softmax(out_d, 1), 1)
    conf_s, pred_s = torch.max(F.softmax(out_s, 1), 1)

for i in range(n_show):
    rgb_img = display_imgs[i].numpy().astype(np.float32) / 255.0
    pd_, td_, cd_ = pred_d[i].item(), disease_lbl[i].item(), conf_d[i].item()
    ps_ = species_names[pred_s[i].item()]
    
    grayscale_cam = cam(input_tensor=inputs_gpu[i:i+1], targets=[ClassifierOutputTarget(pd_)])[0]
    grayscale_cam_resized = cv2.resize(grayscale_cam, (rgb_img.shape[1], rgb_img.shape[0]))
    cam_image = show_cam_on_image(rgb_img, grayscale_cam_resized, use_rgb=True)
    
    axes[i, 0].imshow(rgb_img); axes[i, 0].set_title(f'Asli\nTrue: {class_names[td_]}'); axes[i, 0].axis('off')
    axes[i, 1].imshow(grayscale_cam_resized, cmap='jet'); axes[i, 1].set_title('Heatmap'); axes[i, 1].axis('off')
    
    status = "[BENAR]" if pd_ == td_ else "[SALAH]"
    color = 'green' if pd_ == td_ else 'red'
    axes[i, 2].imshow(cam_image)
    axes[i, 2].set_title(f'{status} {class_names[pd_]} ({cd_*100:.0f}%)\nSpesies: {ps_}', color=color, fontweight='bold', fontsize=9)
    axes[i, 2].axis('off')
    
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['OUTPUT_DIR'], f'gradcam_{CONFIG["BACKBONE"]}.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
class TemperatureScaler(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model
        self.temperature = nn.Parameter(torch.ones(1) * 1.5)

    def forward(self, x):
        disease_out, species_out = self.model(x)
        return disease_out / self.temperature, species_out

    def calibrate(self, val_loader, device, max_iter=50):
        self.to(device)
        logits_list, labels_list = [], []
        self.model.eval()
        with torch.no_grad():
            for inputs, disease_lbl, species_lbl, _, _ in val_loader:
                inputs = inputs.to(device)
                out_d, _ = self.model(inputs)
                logits_list.append(out_d)
                labels_list.append(disease_lbl.to(device))
        logits = torch.cat(logits_list)
        labels = torch.cat(labels_list)
        optimizer = optim.LBFGS([self.temperature], lr=0.01, max_iter=max_iter)
        nll_criterion = nn.CrossEntropyLoss()
        def eval_step():
            optimizer.zero_grad()
            loss = nll_criterion(logits / self.temperature, labels)
            loss.backward()
            return loss
        optimizer.step(eval_step)
        print(f"Temperature terkalibrasi: {self.temperature.item():.3f}")
        return self.temperature.item()

temp_scaler = TemperatureScaler(model_ft)
best_temperature = temp_scaler.calibrate(dataloaders['val'], device)
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'temperature.json'), 'w') as f:
    json.dump({'temperature': best_temperature}, f)

In [ ]:
import torchvision.datasets as tv_datasets

cifar_neg = tv_datasets.CIFAR10(root='/tmp/cifar', train=True, download=True)
NEG_SAMPLE_SIZE = min(800, len(full_dataset))

negative_images = []
neg_indices = random.sample(range(len(cifar_neg)), NEG_SAMPLE_SIZE)
for idx in neg_indices:
    img, _ = cifar_neg[idx]
    negative_images.append(img.resize((CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'])))

print(f"Gambar negatif (bukan daun) terkumpul: {len(negative_images)}")

class LeafGateDataset(Dataset):
    def __init__(self, positive_paths, negative_images, transform):
        self.positive_paths = positive_paths
        self.negative_images = negative_images
        self.transform = transform

    def __len__(self):
        return len(self.positive_paths) + len(self.negative_images)

    def __getitem__(self, idx):
        if idx < len(self.positive_paths):
            img = Image.open(self.positive_paths[idx]).convert('RGB')
            label = 1
        else:
            img = self.negative_images[idx - len(self.positive_paths)].convert('RGB')
            label = 0
        return self.transform(img), label


positive_paths = [full_dataset.image_paths[i] for i in random.sample(train_indices, min(800, len(train_indices)))]

gate_transform = transforms.Compose([
    transforms.Resize((CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'])),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
])

gate_dataset = LeafGateDataset(positive_paths, negative_images, gate_transform)
gate_loader = DataLoader(gate_dataset, batch_size=32, shuffle=True, num_workers=2)

class LeafGateModel(nn.Module):
    def __init__(self, trained_backbone):
        super().__init__()
        self.backbone = trained_backbone
        for p in self.backbone.parameters():
            p.requires_grad = False
        in_features = 1024 if CONFIG['BACKBONE'] == 'densenet121' else 2048
        self.gate_head = nn.Sequential(
            nn.Dropout(0.3), nn.Linear(in_features, 64), nn.ReLU(inplace=True), nn.Linear(64, 1)
        )

    def forward(self, x):
        with torch.no_grad():
            feat = self.backbone(x)
        return self.gate_head(feat).squeeze(1)


gate_model = LeafGateModel(model_ft.base).to(device)
gate_optimizer = optim.AdamW(gate_model.gate_head.parameters(), lr=1e-3)
gate_criterion = nn.BCEWithLogitsLoss()

print("Melatih gate binary classifier (cepat, cuma head kecil)...")
gate_model.train()
for epoch in range(8):
    total_loss, correct, total = 0, 0, 0
    for imgs, labels in gate_loader:
        imgs, labels = imgs.to(device), labels.float().to(device)
        gate_optimizer.zero_grad()
        logits = gate_model(imgs)
        loss = gate_criterion(logits, labels)
        loss.backward()
        gate_optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct += ((torch.sigmoid(logits) > 0.5).float() == labels).sum().item()
        total += imgs.size(0)
    print(f"Epoch {epoch+1}/8 | Loss: {total_loss/total:.4f} | Acc: {correct/total:.4f}")

torch.save(gate_model.gate_head.state_dict(), os.path.join(CONFIG['OUTPUT_DIR'], f'leaf_gate_head_{CONFIG["BACKBONE"]}.pth'))
print("Gate model tersimpan.")

gate_model.eval()
sample_leaf = Image.open(full_dataset.image_paths[test_indices[0]]).convert('RGB')
with torch.no_grad():
    test_input = gate_transform(sample_leaf).unsqueeze(0).to(device)
    prob_leaf = torch.sigmoid(gate_model(test_input)).item()
print(f"\nSanity check — gambar daun asli dari test set: probabilitas 'daun' = {prob_leaf:.4f} (harus mendekati 1.0)")

In [ ]:
def predict_for_website(model, gate_model, pil_image, eval_transform, class_names, species_names,
                         temperature_d=1.0, n_aug=4, device=device, sehat_label='Sehat',
                         gate_threshold=0.5):
    # ===== GERBANG: cek dulu apakah ini gambar daun sama sekali =====
    gate_model.eval()
    with torch.no_grad():
        img_t_gate = eval_transform(pil_image).unsqueeze(0).to(device)
        gate_logit = gate_model(img_t_gate)
        is_leaf_prob = torch.sigmoid(gate_logit).item()

    if is_leaf_prob < gate_threshold:
        return {
            "valid": False,
            "message": "Gambar ini sepertinya bukan foto daun Hoya. Coba upload foto daun yang jelas.",
            "confidence_bukan_daun": round((1 - is_leaf_prob) * 100, 1),
        }
        
    model.eval()
    probs_d_all, probs_s_all = [], []
    with torch.no_grad():
        img_t = eval_transform(pil_image).unsqueeze(0).to(device)
        out_d, out_s = model(img_t)
        probs_d_all.append(F.softmax(out_d / temperature_d, dim=1))
        probs_s_all.append(F.softmax(out_s, dim=1))

        tta_transform = transforms.Compose([
            transforms.Resize((CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE'])),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.1, contrast=0.1),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN.tolist(), IMAGENET_STD.tolist()),
        ])
        for _ in range(n_aug):
            img_t = tta_transform(pil_image).unsqueeze(0).to(device)
            out_d, out_s = model(img_t)
            probs_d_all.append(F.softmax(out_d / temperature_d, dim=1))
            probs_s_all.append(F.softmax(out_s, dim=1))

    avg_d = torch.mean(torch.cat(probs_d_all, 0), 0)
    avg_s = torch.mean(torch.cat(probs_s_all, 0), 0)
    pred_disease_idx = torch.argmax(avg_d).item()
    pred_species_idx = torch.argmax(avg_s).item()
    disease_name = class_names[pred_disease_idx]
    species_name = species_names[pred_species_idx]
    is_healthy = (disease_name == sehat_label)

    top3_idx = torch.topk(avg_d, min(3, len(class_names))).indices.cpu().numpy()
    top3_conf = torch.topk(avg_d, min(3, len(class_names))).values.cpu().numpy()

    top1_conf = float(top3_conf[0])
    top2_conf = float(top3_conf[1]) if len(top3_conf) > 1 else 0
    is_ambiguous = (top1_conf - top2_conf) < 0.15

    return {
        "valid": True,
        "spesies": {"nama": species_name, "confidence": round(float(avg_s[pred_species_idx].item())*100, 1)},
        "gejala": {
            "status": "sehat" if is_healthy else "sakit",
            "nama_tampilan": "Tidak terkendala gejala (sehat)" if is_healthy else disease_name,
            "nama_kelas_model": disease_name,
            "confidence": round(top1_conf*100, 1),
            "is_ambiguous": is_ambiguous,
            "top3": [{"nama": class_names[i], "confidence": round(float(c)*100, 1)} for i, c in zip(top3_idx, top3_conf)],
        },
        "faktor_penyebab": None, "penanganan": None,
    }

sample_path = full_dataset.image_paths[test_indices[0]]
sample_img = Image.open(sample_path).convert('RGB')
hasil = predict_for_website(model_ft, gate_model, sample_img, eval_transform, class_names, species_names, temperature_d=best_temperature)
print(hasil)

In [ ]:
model_ft.eval()
example_input = torch.randn(1, 3, CONFIG['IMG_SIZE'], CONFIG['IMG_SIZE']).to(device)
traced_model = torch.jit.trace(model_ft, example_input)
traced_model.save(os.path.join(CONFIG['OUTPUT_DIR'], f'hoya_multitask_{CONFIG["BACKBONE"]}_traced.pt'))

deployment_meta = {
    'class_names': class_names, 'species_names': species_names,
    'img_size': CONFIG['IMG_SIZE'],
    'normalize_mean': IMAGENET_MEAN.tolist(), 'normalize_std': IMAGENET_STD.tolist(),
    'temperature': best_temperature, 'backbone': CONFIG['BACKBONE'],
    'gate_threshold': 0.5,
}
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'deployment_meta.json'), 'w') as f:
    json.dump(deployment_meta, f, indent=2, ensure_ascii=False)

torch.save(model_ft.state_dict(), os.path.join(CONFIG['OUTPUT_DIR'], f'hoya_multitask_{CONFIG["BACKBONE"]}_final.pth'))

# BARU: simpan gate model (leaf vs non-leaf)
torch.save(gate_model.gate_head.state_dict(), os.path.join(CONFIG['OUTPUT_DIR'], f'leaf_gate_head_{CONFIG["BACKBONE"]}.pth'))
print("Gate model (leaf vs non-leaf) juga tersimpan untuk deployment.")

with open(os.path.join(CONFIG['OUTPUT_DIR'], 'class_names.json'), 'w') as f:
    json.dump(class_names, f, indent=2, ensure_ascii=False)
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'species_names.json'), 'w') as f:
    json.dump(species_names, f, indent=2, ensure_ascii=False)
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'training_history.json'), 'w') as f:
    json.dump(history, f, indent=2)
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'classification_report_disease.json'), 'w') as f:
    json.dump(report_dict_disease, f, indent=2)
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'classification_report_species.json'), 'w') as f:
    json.dump(report_dict_species, f, indent=2)
with open(os.path.join(CONFIG['OUTPUT_DIR'], 'config.json'), 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"Semua artefak tersimpan di: {CONFIG['OUTPUT_DIR']}")
print(sorted(os.listdir(CONFIG['OUTPUT_DIR'])))